<a href="https://colab.research.google.com/github/Maddox159-crypto/ESAA_assignment/blob/main/ob%20%ED%94%84%EB%A1%9C%EC%A0%9D%ED%8A%B81_%EC%B5%9C%EC%A2%85%EC%BD%94%EB%93%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0922 실험 — 토스 CTR 예측 (LightGBM)

## 0. 환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 1. Claude Code 설치
!curl -fsSL https://claude.ai/install.sh | bash

# 2. 오른쪽 터미널에서도 claude를 바로 실행할 수 있도록 bashrc에 경로 등록
!echo 'export PATH="$HOME/.local/bin:$PATH"' >> ~/.bashrc

Setting up Claude Code...
78
Checkinginstallationstatus...
Installing Clude Cde nive build latest...
Seting uplauncher and shellintegration.
⚠ Setup notes:
●Nativeinstallationexistsbut~/.local/binisnotinyourPATH.Run:

echo'exportPATH="$HOME/.local/bin:$PATH"'>>~/.bashrc&&source~/.bashrc

Location: ~/.local/bin/claude


Next:Runclaude--help togetstarted

⚠Setupnotes:
●Nativeinstallationexistsbut~/.local/binisnotinyourPATH.Run:

echo'exportPATH="$HOME/.local/bin:$PATH"'>>~/.bashrc&&source~/.bashrc

(B[>4m[<u78
✅ Installation complete!



In [ ]:
!pip install optuna

## 1. Config & 기본 파이프라인 함수

In [ ]:
"""
토스 NEXT ML Challenge : 광고 클릭 예측(CTR) 모델 개발
------------------------------------------------------
베이스라인 파이프라인
  1) 데이터 로드 (메모리 최적화)
  2) 간단 EDA
  3) 피처 엔지니어링
  4) LightGBM 학습 (Score = 0.5*AP + 0.5*(1/(1+WLL)) 커스텀 평가)
  5) 예측 및 제출 파일 생성

실행 전제:
  - train.parquet, test.parquet 이 DATA_DIR 아래에 있다고 가정
  - pip install lightgbm pyarrow scikit-learn pandas numpy
"""

import os
import gc
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, log_loss
import lightgbm as lgb

# ------------------------------------------------------------------
# 0. Config
# ------------------------------------------------------------------
DATA_DIR = "/content/drive/MyDrive/ESAA"          # train.parquet / test.parquet 위치 (본인 경로로 수정)
TRAIN_PATH = os.path.join(DATA_DIR, "train.parquet")
TEST_PATH = os.path.join(DATA_DIR, "test.parquet")
SAMPLE_SUBMISSION_PATH = os.path.join(DATA_DIR, "sample_submission.csv")  # 대회에서 준 제출 양식 파일
OUTPUT_PATH = os.path.join(DATA_DIR, "submission.csv")  # 파일명은 제출 전에 원하는 이름으로 바꿔도 됨

TARGET_COL = "clicked"
ID_COL = "ID"
SEED = 42

# scale_pos_weight 적용 여부 실험용 플래그
# 실험 결과: False로 끄니 Score 0.354 -> 0.2226으로 급락 (AP 붕괴) -> 반드시 True로 유지할 것
USE_SCALE_POS_WEIGHT = True

# 하이퍼파라미터 - 여기만 바꾸고 이 셀 + main() 재실행하면 빠르게 실험 가능
# (train_model 함수 자체는 다시 안 건드려도 됨) -> Optuna 튜닝 결과가 자동으로 이 값들을 갱신함
# 2026-09-23: Optuna 튜닝값이 day7 검증에서 오히려 0914(기본값) 기록을 못 넘어서
#             day7 방식으로 기본값과 재비교 실험 중 -> 아래는 원래 기본값
NUM_LEAVES = 63
LEARNING_RATE = 0.05
MIN_DATA_IN_LEAF = 100
FEATURE_FRACTION = 0.8
BAGGING_FRACTION = 0.8
LAMBDA_L1 = 0.0
LAMBDA_L2 = 0.0
NUM_BOOST_ROUND = 2000

# 큰 데이터로 빠르게 파이프라인 검증할 때 쓰는 샘플링 비율 (1.0 = 전체 사용)
TRAIN_SAMPLE_FRAC = 1.0

# ------------------------------------------------------------------
# 1. 데이터 로드 + 메모리 최적화
# ------------------------------------------------------------------
def reduce_mem_usage(df: pd.DataFrame, cat_cols=None) -> pd.DataFrame:
    """수치형 컬럼 다운캐스팅 + 지정 컬럼 category화로 메모리 절약."""
    cat_cols = cat_cols or []
    start_mem = df.memory_usage(deep=True).sum() / 1024**2

    for col in df.columns:
        if col in cat_cols:
            # gender/age_group 등이 1.0, 2.0 같은 float 코드로 들어있는 경우가 많아서
            # category화 전에 정수로 먼저 정리 (결측은 -1 같은 보초값으로 대체 후 정수화)
            if pd.api.types.is_float_dtype(df[col]):
                df[col] = df[col].fillna(-1).astype(np.int32)
            df[col] = df[col].astype("category")
            continue

        col_type = df[col].dtype
        if col_type == object:
            continue

        c_min, c_max = df[col].min(), df[col].max()
        if pd.api.types.is_integer_dtype(col_type):
            if c_min >= 0:
                if c_max < np.iinfo(np.uint8).max:
                    df[col] = df[col].astype(np.uint8)
                elif c_max < np.iinfo(np.uint16).max:
                    df[col] = df[col].astype(np.uint16)
                elif c_max < np.iinfo(np.uint32).max:
                    df[col] = df[col].astype(np.uint32)
            else:
                if np.iinfo(np.int8).min < c_min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif np.iinfo(np.int16).min < c_min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif np.iinfo(np.int32).min < c_min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
        else:
            df[col] = df[col].astype(np.float32)

    end_mem = df.memory_usage(deep=True).sum() / 1024**2
    print(f"[reduce_mem_usage] {start_mem:.1f} MB -> {end_mem:.1f} MB "
          f"({100*(start_mem-end_mem)/start_mem:.1f}% 감소)")
    return df


def get_feature_groups(columns):
    """컬럼명 prefix 기준으로 피처 그룹 자동 분류."""
    groups = {
        "explicit": [],   # gender, age_group, inventory_id, day_of_week, hour, seq
        "l_feat": [],
        "feat_a": [], "feat_b": [], "feat_c": [], "feat_d": [], "feat_e": [],
        "history_a": [],
    }
    explicit_names = {"gender", "age_group", "inventory_id", "day_of_week", "hour", "seq"}
    for col in columns:
        if col in explicit_names:
            groups["explicit"].append(col)
        elif col.startswith("l_feat_"):
            groups["l_feat"].append(col)
        elif col.startswith("feat_a_"):
            groups["feat_a"].append(col)
        elif col.startswith("feat_b_"):
            groups["feat_b"].append(col)
        elif col.startswith("feat_c_"):
            groups["feat_c"].append(col)
        elif col.startswith("feat_d_"):
            groups["feat_d"].append(col)
        elif col.startswith("feat_e_"):
            groups["feat_e"].append(col)
        elif col.startswith("history_a_"):
            groups["history_a"].append(col)
    return groups


def load_data():
    print("=" * 60)
    print("[1] 데이터 로드")
    print("=" * 60)
    train = pd.read_parquet(TRAIN_PATH)
    test = pd.read_parquet(TEST_PATH)

    if TRAIN_SAMPLE_FRAC < 1.0:
        train = train.sample(frac=TRAIN_SAMPLE_FRAC, random_state=SEED).reset_index(drop=True)
        print(f"  -> 빠른 검증을 위해 train {TRAIN_SAMPLE_FRAC*100:.0f}% 샘플링")

    print(f"train shape: {train.shape}, test shape: {test.shape}")

    # hour도 explicit 그룹인데 원본 parquet에서 object(문자열)로 들어오는 경우가 있어서
    # gender/age_group과 동일하게 category 변환 목록에 포함 (그냥 두면 raw로 남아 LightGBM에서 dtype 에러)
    # l_feat_14는 categorical로 실험해봤으나 성능 하락 확인 (Score 0.35402->0.35093) -> 수치형으로 원복
    cat_cols = ["gender", "age_group", "inventory_id", "day_of_week", "hour"]
    cat_cols = [c for c in cat_cols if c in train.columns]

    train = reduce_mem_usage(train, cat_cols=cat_cols)
    test = reduce_mem_usage(test, cat_cols=[c for c in cat_cols if c in test.columns])

    return train, test


# ------------------------------------------------------------------
# 2. 간단 EDA
# ------------------------------------------------------------------
def run_eda(train: pd.DataFrame):
    print("\n" + "=" * 60)
    print("[2] EDA")
    print("=" * 60)

    print("\n-- Target 분포 --")
    print(train[TARGET_COL].value_counts(normalize=True))

    print("\n-- 결측치 상위 15개 컬럼 --")
    na_ratio = train.isna().mean().sort_values(ascending=False)
    print(na_ratio.head(15))

    print("\n-- explicit 피처 카디널리티 --")
    groups = get_feature_groups(train.columns)
    for col in groups["explicit"]:
        print(f"  {col}: n_unique={train[col].nunique()}")

    print("\n-- 피처 그룹별 컬럼 수 --")
    for k, v in groups.items():
        print(f"  {k}: {len(v)}개")

    return groups


# ------------------------------------------------------------------
# 3. 피처 엔지니어링
# ------------------------------------------------------------------
def parse_seq_features(df: pd.DataFrame, col: str = "seq") -> pd.DataFrame:
    """
    'seq' 같은 콤마 구분 시퀀스 문자열(예: "9,18,269,516,...")에서
    요약 통계 피처를 벡터화 연산으로 추출.
    전체 원소 파싱(mean/std 등)은 10M 행 기준 느릴 수 있어 우선 len/first/last만 사용.
    """
    s = df[col].astype(str)

    df[f"{col}_len"] = (s.str.count(",") + 1).astype(np.int32)
    df[f"{col}_len_log1p"] = np.log1p(df[f"{col}_len"]).astype(np.float32)
    df[f"{col}_first"] = pd.to_numeric(
        s.str.split(",", n=1).str[0], errors="coerce"
    ).astype(np.float32)
    df[f"{col}_last"] = pd.to_numeric(
        s.str.rsplit(",", n=1).str[-1], errors="coerce"
    ).astype(np.float32)

    return df


def build_features(train: pd.DataFrame, test: pd.DataFrame, groups: dict):
    print("\n" + "=" * 60)
    print("[3] 피처 엔지니어링")
    print("=" * 60)

    feature_cols = []
    for k in ["explicit", "l_feat", "feat_a", "feat_b", "feat_c", "feat_d", "feat_e", "history_a"]:
        feature_cols.extend(groups[k])

    # seq는 단일 숫자가 아니라 "9,18,269,516,..." 형태의 콤마 구분 로그 시퀀스 문자열.
    # 그대로 float 변환하면 에러나므로 원본은 피처에서 빼고, 요약 통계만 뽑아서 사용.
    if "seq" in feature_cols:
        feature_cols.remove("seq")
        for df in (train, test):
            parse_seq_features(df, col="seq")
        seq_derived = ["seq_len", "seq_len_log1p", "seq_first", "seq_last"]
        feature_cols.extend(seq_derived)

    # hour를 순환형(sin/cos)으로도 인코딩 -> 트리 모델엔 필수는 아니지만 시도해볼 가치 있음
    if "hour" in feature_cols:
        for df in (train, test):
            df["hour_sin"] = np.sin(2 * np.pi * df["hour"].astype(np.float32) / 24)
            df["hour_cos"] = np.cos(2 * np.pi * df["hour"].astype(np.float32) / 24)
        feature_cols.extend(["hour_sin", "hour_cos"])

    # history_a_1/2/3가 피처 중요도 최상위권 -> 서로 비율/차이 조합 피처 추가
    # (단독으로 강력한 피처들일수록 상호작용에서 추가 신호가 나올 가능성이 높음)
    history_top = [c for c in ["history_a_1", "history_a_2", "history_a_3"] if c in feature_cols]
    if len(history_top) >= 2:
        eps = 1e-6
        for i in range(len(history_top)):
            for j in range(i + 1, len(history_top)):
                a, b = history_top[i], history_top[j]
                ratio_col = f"{a}_div_{b}"
                diff_col = f"{a}_minus_{b}"
                for df in (train, test):
                    df[ratio_col] = (df[a] / (df[b].astype(np.float32) + eps)).astype(np.float32)
                    df[diff_col] = (df[a].astype(np.float32) - df[b].astype(np.float32)).astype(np.float32)
                feature_cols.extend([ratio_col, diff_col])
        print(f"  history_a 조합 피처 {2 * len(history_top) * (len(history_top) - 1) // 2}개 추가")

    cat_features = [c for c in ["gender", "age_group", "inventory_id", "day_of_week", "hour"] if c in feature_cols]

    print(f"  최종 사용 피처 수: {len(feature_cols)}개")
    print(f"  범주형 피처: {cat_features}")

    return feature_cols, cat_features


# ------------------------------------------------------------------
# 4. 커스텀 평가지표: Score = 0.5*AP + 0.5*(1/(1+WLL))
# ------------------------------------------------------------------
def weighted_logloss(y_true, y_pred, eps=1e-15):
    """clicked=0과 1의 클래스 기여를 50:50으로 맞춘 가중 LogLoss."""
    y_pred = np.clip(y_pred, eps, 1 - eps)
    pos_mask = y_true == 1
    neg_mask = y_true == 0

    ll_pos = -np.log(y_pred[pos_mask]).mean() if pos_mask.sum() > 0 else 0.0
    ll_neg = -np.log(1 - y_pred[neg_mask]).mean() if neg_mask.sum() > 0 else 0.0

    return 0.5 * ll_pos + 0.5 * ll_neg


def competition_score(y_true, y_pred):
    ap = average_precision_score(y_true, y_pred)
    wll = weighted_logloss(y_true, y_pred)
    score = 0.5 * ap + 0.5 * (1.0 / (1.0 + wll))
    return score, ap, wll


def lgb_competition_eval(y_pred, dataset):
    """LightGBM custom eval용 wrapper (값이 클수록 좋음)."""
    y_true = dataset.get_label()
    score, ap, wll = competition_score(y_true, y_pred)
    return "comp_score", score, True  # True = higher is better


# ------------------------------------------------------------------
# 5. 학습
# ------------------------------------------------------------------
def train_model(train: pd.DataFrame, feature_cols, cat_features):
    print("\n" + "=" * 60)
    print("[4] 모델 학습")
    print("=" * 60)

    X = train[feature_cols]
    y = train[TARGET_COL].astype(np.int8)

    X_train, X_valid, y_train, y_valid = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )

    train_set = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_features, free_raw_data=False)
    valid_set = lgb.Dataset(X_valid, label=y_valid, categorical_feature=cat_features, reference=train_set, free_raw_data=False)

    # 클래스 불균형이 클 것으로 예상 (CTR 데이터 특성상 클릭=1이 소수)
    pos = (y_train == 1).sum()
    neg = (y_train == 0).sum()
    # WLL 산식 자체가 이미 0/1 클래스를 50:50으로 맞춰주기 때문에,
    # scale_pos_weight로 확률을 인위적으로 밀어올리면 이중 보정 -> calibration이 깨져 WLL엔 오히려 손해일 수 있음
    # USE_SCALE_POS_WEIGHT=False로 실험해서 True(기존)와 비교해볼 것
    if USE_SCALE_POS_WEIGHT:
        scale_pos_weight = neg / max(pos, 1)
    else:
        scale_pos_weight = 1.0
    print(f"  pos={pos}, neg={neg}, scale_pos_weight={scale_pos_weight:.2f} (USE_SCALE_POS_WEIGHT={USE_SCALE_POS_WEIGHT})")

    params = {
        "objective": "binary",
        "metric": "None",          # custom eval 사용
        "learning_rate": LEARNING_RATE,
        "num_leaves": NUM_LEAVES,
        "max_depth": -1,
        "min_data_in_leaf": MIN_DATA_IN_LEAF,
        "feature_fraction": FEATURE_FRACTION,
        "bagging_fraction": BAGGING_FRACTION,
        "bagging_freq": 1,
        "lambda_l1": LAMBDA_L1,
        "lambda_l2": LAMBDA_L2,
        "scale_pos_weight": scale_pos_weight,
        "seed": SEED,
        "verbosity": -1,
    }

    model = lgb.train(
        params,
        train_set,
        num_boost_round=NUM_BOOST_ROUND,
        valid_sets=[valid_set],
        feval=lgb_competition_eval,
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=50),
        ],
    )

    val_pred = model.predict(X_valid, num_iteration=model.best_iteration)
    score, ap, wll = competition_score(y_valid.values, val_pred)
    print(f"\n[Validation] Score={score:.5f}  AP={ap:.5f}  WLL={wll:.5f}")

    return model


# ------------------------------------------------------------------
# 6. 예측 및 제출 파일 생성
# ------------------------------------------------------------------
def predict_and_save(model, test: pd.DataFrame, feature_cols):
    print("\n" + "=" * 60)
    print("[5] 예측 및 제출 파일 생성")
    print("=" * 60)

    test_pred = model.predict(test[feature_cols], num_iteration=model.best_iteration)

    if os.path.exists(SAMPLE_SUBMISSION_PATH):
        # 제출 양식 파일이 있으면 그 컬럼명/순서/ID 순서를 그대로 따라감
        sample_sub = pd.read_csv(SAMPLE_SUBMISSION_PATH)
        print(f"  -> 제출 양식 컬럼: {list(sample_sub.columns)}")

        pred_map = pd.Series(test_pred, index=test[ID_COL].values)
        target_col_in_sample = [c for c in sample_sub.columns if c != ID_COL][0]

        submission = sample_sub.copy()
        submission[target_col_in_sample] = submission[ID_COL].map(pred_map)

        assert submission[target_col_in_sample].isna().sum() == 0, \
            "양식의 ID 중 예측값이 매핑 안 된 행이 있음 -> ID_COL/test 정렬 확인 필요"
    else:
        # 양식 파일이 없으면 기본 포맷으로 생성
        submission = pd.DataFrame({
            ID_COL: test[ID_COL] if ID_COL in test.columns else np.arange(len(test)),
            TARGET_COL: test_pred,
        })

    submission.to_csv(OUTPUT_PATH, index=False)
    print(f"  -> {OUTPUT_PATH} 저장 완료 (shape={submission.shape})")

## 2. day7 검증 / seed 앙상블 / 전체 데이터 최종 학습 함수

In [ ]:
# 전역변수 의존성 버그 방지용 명시적 기본값 (train_model_day7_val이 인자로 안 받으면 이걸 씀)
DEFAULT_LGB_PARAMS = {
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "lambda_l1": 0.0,
    "lambda_l2": 0.0,
}


# ------------------------------------------------------------------
# 핵심 발견: test는 전부 day_of_week==7. 랜덤 split은 test 상황을 대변 못함.
# -> validation을 day_of_week==7인 행으로만 구성해서 test 조건을 실제로 흉내냄
#
# 버그 수정 (2026-09-23): 예전 버전은 NUM_LEAVES 등 전역변수를 읽었는데,
# apply_best_params_and_retrain이 global로 이 값들을 영구 덮어써서
# "기본값으로 되돌렸다고 생각했는데 실제론 튜닝값이 남아있는" 사고가 남.
# -> 이제 params를 함수 인자로 명시적으로 받고, 실제 쓰인 값을 항상 출력함.
# ------------------------------------------------------------------
def train_model_day7_val(train_df: pd.DataFrame, feature_cols, cat_features,
                          lgb_params=None, num_boost_round=None, seed=None):
    print("\n" + "=" * 60)
    print("[4-D7] 모델 학습 (day_of_week==7 기준 validation)")
    print("=" * 60)

    lgb_params = dict(DEFAULT_LGB_PARAMS) if lgb_params is None else dict(lgb_params)
    nbr = NUM_BOOST_ROUND if num_boost_round is None else num_boost_round
    run_seed = SEED if seed is None else seed
    print(f"  실제 사용 파라미터: {lgb_params}")
    print(f"  num_boost_round={nbr}, seed={run_seed}")

    val_mask = train_df["day_of_week"].astype(str) == "7"
    print(f"  train(1~6요일): {(~val_mask).sum():,}  /  valid(7요일): {val_mask.sum():,}")

    X_train = train_df.loc[~val_mask, feature_cols]
    y_train = train_df.loc[~val_mask, TARGET_COL].astype(np.int8)
    X_valid = train_df.loc[val_mask, feature_cols]
    y_valid = train_df.loc[val_mask, TARGET_COL].astype(np.int8)

    train_set = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_features, free_raw_data=False)
    valid_set = lgb.Dataset(X_valid, label=y_valid, categorical_feature=cat_features, reference=train_set, free_raw_data=False)

    pos = (y_train == 1).sum()
    neg = (y_train == 0).sum()
    scale_pos_weight = (neg / max(pos, 1)) if USE_SCALE_POS_WEIGHT else 1.0
    print(f"  pos={pos}, neg={neg}, scale_pos_weight={scale_pos_weight:.2f}")

    params = {
        "objective": "binary",
        "metric": "None",
        "max_depth": -1,
        "bagging_freq": 1,
        "scale_pos_weight": scale_pos_weight,
        "seed": run_seed,
        "verbosity": -1,
        **lgb_params,
    }

    model = lgb.train(
        params,
        train_set,
        num_boost_round=nbr,
        valid_sets=[valid_set],
        feval=lgb_competition_eval,
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=50),
        ],
    )

    val_pred = model.predict(X_valid, num_iteration=model.best_iteration)
    score, ap, wll = competition_score(y_valid.values, val_pred)
    print(f"\n[day7 Validation] Score={score:.5f}  AP={ap:.5f}  WLL={wll:.5f}")

    return model


# ------------------------------------------------------------------
# seed 앙상블: 같은 파라미터, 다른 seed로 N개 모델 학습 후 예측 평균
# (day_of_week==7 validation split은 seed 무관하게 고정 -> 순수하게 모델의
#  랜덤성(bagging, feature sampling 등)만 다양화해서 분산을 줄이는 효과)
# ------------------------------------------------------------------
def train_seed_ensemble(train_df: pd.DataFrame, feature_cols, cat_features,
                         lgb_params, n_seeds=5, base_seed=100, num_boost_round=None):
    models = []
    val_mask = train_df["day_of_week"].astype(str) == "7"
    y_valid = train_df.loc[val_mask, TARGET_COL].astype(np.int8).values
    oof_preds = []

    for i in range(n_seeds):
        seed = base_seed + i
        print(f"\n{'#' * 60}\n# Seed {seed} ({i+1}/{n_seeds})\n{'#' * 60}")
        model = train_model_day7_val(train_df, feature_cols, cat_features,
                                      lgb_params=lgb_params, num_boost_round=num_boost_round, seed=seed)
        models.append(model)
        val_pred = model.predict(train_df.loc[val_mask, feature_cols], num_iteration=model.best_iteration)
        oof_preds.append(val_pred)

    ensemble_pred = np.mean(oof_preds, axis=0)
    score, ap, wll = competition_score(y_valid, ensemble_pred)
    print("\n" + "=" * 60)
    print(f"[Seed 앙상블 {n_seeds}개 평균] Score={score:.5f}  AP={ap:.5f}  WLL={wll:.5f}")
    print("=" * 60)

    return models


def predict_and_save_seed_ensemble(models, test: pd.DataFrame, feature_cols):
    print("\n" + "=" * 60)
    print("[5-SE] 예측 및 제출 파일 생성 (seed 앙상블 평균)")
    print("=" * 60)

    fold_preds = [m.predict(test[feature_cols], num_iteration=m.best_iteration) for m in models]
    test_pred = np.mean(fold_preds, axis=0)

    if os.path.exists(SAMPLE_SUBMISSION_PATH):
        sample_sub = pd.read_csv(SAMPLE_SUBMISSION_PATH)
        pred_map = pd.Series(test_pred, index=test[ID_COL].values)
        target_col_in_sample = [c for c in sample_sub.columns if c != ID_COL][0]

        submission = sample_sub.copy()
        submission[target_col_in_sample] = submission[ID_COL].map(pred_map)
        assert submission[target_col_in_sample].isna().sum() == 0
    else:
        submission = pd.DataFrame({
            ID_COL: test[ID_COL] if ID_COL in test.columns else np.arange(len(test)),
            TARGET_COL: test_pred,
        })

    submission.to_csv(OUTPUT_PATH, index=False)
    print(f"  -> {OUTPUT_PATH} 저장 완료 (shape={submission.shape})")


# ------------------------------------------------------------------
# 최종 production 학습: day7로 찾은 최적 파라미터 + best_iteration을 그대로 쓰되,
# day7 데이터도 포함한 "전체" train으로 다시 학습 (실전에서 day7 패턴을 놓치지 않기 위함)
# early stopping용 valid set이 없으니 num_boost_round를 고정값으로 줘야 함
# (day7 검증에서 나온 model.best_iteration을 그대로 넘기면 됨)
# ------------------------------------------------------------------
def train_final_full_data(train_df: pd.DataFrame, feature_cols, cat_features,
                           lgb_params, num_boost_round, seed=None):
    print("\n" + "=" * 60)
    print(f"[FINAL] 전체 데이터(day7 포함)로 최종 학습, num_boost_round={num_boost_round}")
    print("=" * 60)

    run_seed = SEED if seed is None else seed
    X = train_df[feature_cols]
    y = train_df[TARGET_COL].astype(np.int8)

    pos = (y == 1).sum()
    neg = (y == 0).sum()
    scale_pos_weight = (neg / max(pos, 1)) if USE_SCALE_POS_WEIGHT else 1.0
    print(f"  전체 pos={pos}, neg={neg}, scale_pos_weight={scale_pos_weight:.2f}, seed={run_seed}")

    train_set = lgb.Dataset(X, label=y, categorical_feature=cat_features, free_raw_data=False)

    params = {
        "objective": "binary",
        "metric": "None",
        "max_depth": -1,
        "bagging_freq": 1,
        "scale_pos_weight": scale_pos_weight,
        "seed": run_seed,
        "verbosity": -1,
        **lgb_params,
    }

    model = lgb.train(
        params,
        train_set,
        num_boost_round=num_boost_round,  # early stopping 없음 -> valid set이 없어서 고정 라운드
        callbacks=[lgb.log_evaluation(period=100)],
    )
    return model

## 3. Optuna 튜닝 함수

In [ ]:
import optuna
# optuna 버전에 따라 optuna.integration.LightGBMPruningCallback이 별도 패키지(optuna-integration)로
# 분리돼 ModuleNotFoundError가 날 수 있어서, 외부 의존성 없이 pruning 콜백을 직접 구현
def make_pruning_callback(trial, metric_name="comp_score"):
    def _callback(env):
        current_score = None
        for item in env.evaluation_result_list:
            if item[1] == metric_name:
                current_score = item[2]
                break
        if current_score is not None:
            trial.report(current_score, step=env.iteration)
            if trial.should_prune():
                raise optuna.TrialPruned()
    _callback.order = 30
    return _callback

# 튜닝 전용 샘플링 비율 - 전체 데이터로 trial 여러 번 돌리면 너무 오래 걸려서
# 탐색은 일부만으로 하고, 최적 파라미터 찾으면 전체로 최종 학습
TUNING_SAMPLE_FRAC = 0.3
N_TRIALS = 30


def prepare_tuning_data(train: pd.DataFrame, feature_cols, cat_features):
    """
    튜닝용 서브샘플로 day_of_week==7 기준 train/valid split 준비.
    (매 trial 동일 데이터로 비교해야 공정하니 고정)
    2026-09-23: 랜덤 split 기준 튜닝값이 실제 LB에서 원본(기본값)보다 낮게 나와서
                day7 기준으로 재튜닝 -> test가 전부 day_of_week==7이라 이 조건에 맞춰야 함
    """
    if TUNING_SAMPLE_FRAC < 1.0:
        sub = train.sample(frac=TUNING_SAMPLE_FRAC, random_state=SEED)
    else:
        sub = train

    val_mask = sub["day_of_week"].astype(str) == "7"
    X_train = sub.loc[~val_mask, feature_cols]
    y_train = sub.loc[~val_mask, TARGET_COL].astype(np.int8)
    X_valid = sub.loc[val_mask, feature_cols]
    y_valid = sub.loc[val_mask, TARGET_COL].astype(np.int8)
    print(f"  튜닝 데이터 (서브샘플 {TUNING_SAMPLE_FRAC*100:.0f}%): "
          f"train(1~6요일)={len(X_train):,}  valid(7요일)={len(X_valid):,}")

    return X_train, X_valid, y_train, y_valid


def objective(trial, X_train, y_train, X_valid, y_valid, cat_features):
    params = {
        "objective": "binary",
        "metric": "None",
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 31, 255),
        "max_depth": -1,
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 20, 300),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
        "bagging_freq": 1,
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "seed": SEED,
        "verbosity": -1,
    }

    if USE_SCALE_POS_WEIGHT:
        params["scale_pos_weight"] = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
    else:
        params["scale_pos_weight"] = 1.0

    train_set = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_features, free_raw_data=False)
    valid_set = lgb.Dataset(X_valid, label=y_valid, categorical_feature=cat_features, reference=train_set, free_raw_data=False)

    pruning_callback = make_pruning_callback(trial)

    model = lgb.train(
        params,
        train_set,
        num_boost_round=1000,
        valid_sets=[valid_set],
        feval=lgb_competition_eval,
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=0),  # 로그 끔 (trial 많아서 너무 길어짐)
            pruning_callback,
        ],
    )

    val_pred = model.predict(X_valid, num_iteration=model.best_iteration)
    score, ap, wll = competition_score(y_valid.values, val_pred)
    return score


def run_tuning(train: pd.DataFrame, feature_cols, cat_features, n_trials=N_TRIALS):
    print(f"튜닝용 데이터: 전체의 {TUNING_SAMPLE_FRAC*100:.0f}%로 {n_trials}회 trial 진행")

    X_train, X_valid, y_train, y_valid = prepare_tuning_data(train, feature_cols, cat_features)

    study = optuna.create_study(
        direction="maximize",
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=100),
    )
    study.optimize(
        lambda trial: objective(trial, X_train, y_train, X_valid, y_valid, cat_features),
        n_trials=n_trials,
        show_progress_bar=True,
    )

    print("\n" + "=" * 60)
    print(f"Best Score: {study.best_value:.5f}")
    print("Best params:")
    for k, v in study.best_params.items():
        print(f"  {k}: {v}")
    print("=" * 60)

    return study


def apply_best_params_and_retrain(study, train: pd.DataFrame, feature_cols, cat_features):
    """
    Optuna가 찾은 최적 파라미터를 전체 데이터(train)에 적용해서 최종 학습.
    버그 수정: 이제 전역변수를 건드리지 않고 params 딕셔너리를 직접 넘김
    -> 이후 다른 호출(model_day7_default 등)에 몰래 영향 안 줌.
    """
    best = study.best_params
    lgb_params = {
        "num_leaves": best["num_leaves"],
        "learning_rate": best["learning_rate"],
        "min_data_in_leaf": best["min_data_in_leaf"],
        "feature_fraction": best["feature_fraction"],
        "bagging_fraction": best["bagging_fraction"],
        "lambda_l1": best["lambda_l1"],
        "lambda_l2": best["lambda_l2"],
    }
    print(f"튜닝된 파라미터로 학습: {lgb_params}")

    model = train_model_day7_val(train, feature_cols, cat_features, lgb_params=lgb_params)
    return model

## 4. Target encoding 함수

In [ ]:
from sklearn.model_selection import KFold

# target encoding 적용할 컬럼들
# - 단일 컬럼: "inventory_id", "l_feat_14" (고카디널리티 -> 클릭률 자체를 피처로)
# - 조합(interaction): ["inventory_id", "hour"] 처럼 리스트로 넣으면 두 컬럼 조합의 클릭률
TARGET_ENCODE_COLS = [
    "inventory_id",
    "l_feat_14",
    ["inventory_id", "hour"],
    ["inventory_id", "day_of_week"],
]
TE_N_SPLITS = 5
TE_SMOOTHING = 20  # 카디널리티 큰 컬럼에서 표본 적은 카테고리가 극단값 갖는 것 방지


def add_target_encoding(train: pd.DataFrame, test: pd.DataFrame, cols_list, target_col="clicked",
                         n_splits=TE_N_SPLITS, smoothing=TE_SMOOTHING, seed=SEED):
    """
    Out-of-fold target encoding: train은 K-fold로 나눠서 '자기 자신이 속한 fold의 정보는
    안 쓰고' 나머지 fold 통계로 인코딩 (안 그러면 target 정보가 새서 리키지가 됨).
    test는 train 전체 통계로 인코딩 (test는 라벨 모르니 리키지 걱정 없음).
    smoothing으로 카디널리티 큰 컬럼에서 표본 적은 카테고리를 global_mean 쪽으로 당겨줌.
    """
    global_mean = train[target_col].mean()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    y = train[target_col].values
    new_cols = []

    for cols in cols_list:
        if isinstance(cols, str):
            cols = [cols]
        name = "_x_".join(cols)
        oof_col = f"te_{name}"
        new_cols.append(oof_col)

        if len(cols) == 1:
            key_train = train[cols[0]].astype(str)
            key_test = test[cols[0]].astype(str)
        else:
            key_train = train[cols].astype(str).agg("_".join, axis=1)
            key_test = test[cols].astype(str).agg("_".join, axis=1)

        oof = np.full(len(train), np.nan, dtype=np.float32)

        for tr_idx, val_idx in kf.split(train):
            tr_key = key_train.iloc[tr_idx]
            tr_y = y[tr_idx]
            stats = pd.DataFrame({"key": tr_key.values, "y": tr_y}).groupby("key")["y"].agg(["mean", "count"])
            smooth = (stats["mean"] * stats["count"] + global_mean * smoothing) / (stats["count"] + smoothing)
            oof[val_idx] = key_train.iloc[val_idx].map(smooth).values

        oof = pd.Series(oof).fillna(global_mean).astype(np.float32).values
        train[oof_col] = oof

        full_stats = pd.DataFrame({"key": key_train.values, "y": y}).groupby("key")["y"].agg(["mean", "count"])
        full_smooth = (full_stats["mean"] * full_stats["count"] + global_mean * smoothing) / (full_stats["count"] + smoothing)
        test[oof_col] = key_test.map(full_smooth).fillna(global_mean).astype(np.float32)

        print(f"  {oof_col} 추가 완료 (cardinality={len(full_stats)}, "
              f"range=[{train[oof_col].min():.4f}, {train[oof_col].max():.4f}])")

    return train, test, new_cols

## 5. 데이터 로드 · EDA · 피처 생성

In [ ]:
train, test = load_data()
groups = run_eda(train)
feature_cols, cat_features = build_features(train, test, groups)

[1] 데이터 로드
train shape: (10704179, 119), test shape: (1527298, 119)
[reduce_mem_usage] 27897.4 MB -> 25308.0 MB (9.3% 감소)
[reduce_mem_usage] 4127.1 MB -> 3761.9 MB (8.8% 감소)

[2] EDA

-- Target 분포 --
clicked
0    0.980925
1    0.019075
Name: proportion, dtype: float64

-- 결측치 상위 15개 컬럼 --
feat_e_3     0.101414
feat_a_3     0.001737
feat_a_4     0.001737
feat_a_5     0.001737
feat_a_6     0.001737
feat_a_18    0.001737
feat_a_17    0.001737
feat_a_9     0.001737
feat_a_10    0.001737
feat_a_11    0.001737
feat_a_12    0.001737
feat_a_13    0.001737
feat_a_14    0.001737
feat_a_15    0.001737
feat_a_16    0.001737
dtype: float64

-- explicit 피처 카디널리티 --
  gender: n_unique=2
  age_group: n_unique=8
  inventory_id: n_unique=18
  day_of_week: n_unique=7
  hour: n_unique=24
  seq: n_unique=7179942

-- 피처 그룹별 컬럼 수 --
  explicit: 6개
  l_feat: 27개
  feat_a: 18개
  feat_b: 6개
  feat_c: 8개
  feat_d: 6개
  feat_e: 10개
  history_a: 7개

[3] 피처 엔지니어링


/tmp/ipykernel_52693/2425015889.py:195: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_len"] = (s.str.count(",") + 1).astype(np.int32)
/tmp/ipykernel_52693/2425015889.py:196: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_len_log1p"] = np.log1p(df[f"{col}_len"]).astype(np.float32)
/tmp/ipykernel_52693/2425015889.py:197: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd

  history_a 조합 피처 6개 추가
  최종 사용 피처 수: 99개
  범주형 피처: ['gender', 'age_group', 'inventory_id', 'day_of_week', 'hour']


/tmp/ipykernel_52693/2425015889.py:243: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[ratio_col] = (df[a] / (df[b].astype(np.float32) + eps)).astype(np.float32)
/tmp/ipykernel_52693/2425015889.py:244: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[diff_col] = (df[a].astype(np.float32) - df[b].astype(np.float32)).astype(np.float32)


## 6. 진단

### 6-1. train/test 분포 비교 + 홀드아웃 분리

In [ ]:
# ------------------------------------------------------------------
# 진단 1: train vs test의 day_of_week, hour 분포 비교
# -> 다르면 진짜 시간적 괴리(분포 shift) 있다는 뜻
# ------------------------------------------------------------------
print("day_of_week 분포 비교:")
compare_dow = pd.DataFrame({
    "train": train["day_of_week"].value_counts(normalize=True).sort_index(),
    "test": test["day_of_week"].value_counts(normalize=True).sort_index(),
})
compare_dow["diff"] = (compare_dow["train"] - compare_dow["test"]).abs()
print(compare_dow)

print("\nhour 분포 비교:")
compare_hour = pd.DataFrame({
    "train": train["hour"].value_counts(normalize=True).sort_index(),
    "test": test["hour"].value_counts(normalize=True).sort_index(),
})
compare_hour["diff"] = (compare_hour["train"] - compare_hour["test"]).abs()
print(compare_hour)


# ------------------------------------------------------------------
# 진단 2 (더 중요한 것): 우리가 겪은 진짜 문제는 아마 "같은 validation set을
# 30번 넘게 재사용하면서 튜닝 = 그 validation set 자체에 과적합"일 가능성이 높음.
# 그래서 지금부터는 '한 번도 안 건드린 홀드아웃'을 따로 떼어놓고,
# 이후 모든 실험은 거기 손 안 대다가 최종 후보 1~2개만 딱 한 번 확인하는 방식으로 감.
# ------------------------------------------------------------------
from sklearn.model_selection import train_test_split as _tts

HOLDOUT_FRAC = 0.1  # 전체 train의 10%를 최종 검증 전용으로 격리

train_work, train_holdout = _tts(
    train, test_size=HOLDOUT_FRAC, random_state=999,  # 기존 실험에 쓴 seed=42와 다른 seed 사용
    stratify=train[TARGET_COL]
)
print(f"\ntrain_work: {train_work.shape}, train_holdout: {train_holdout.shape} (앞으로 이 홀드아웃은 최종 확인 전까지 절대 안 씀)")

day_of_week 분포 비교:
                train  test      diff
day_of_week                          
1            0.142513   NaN       NaN
2            0.143124   NaN       NaN
3            0.142927   NaN       NaN
4            0.142963   NaN       NaN
5            0.142920   NaN       NaN
6            0.142961   NaN       NaN
7            0.142592   1.0  0.857408

hour 분포 비교:
         train      test      diff
hour                              
00    0.050135  0.064321  0.014186
01    0.016158  0.021068  0.004910
02    0.009218  0.011897  0.002679
03    0.007348  0.009259  0.001912
04    0.010277  0.012083  0.001806
05    0.025522  0.027138  0.001616
06    0.044628  0.039864  0.004764
07    0.056335  0.047769  0.008566
08    0.070562  0.063285  0.007277
09    0.054879  0.059465  0.004586
10    0.053851  0.059535  0.005683
11    0.047899  0.049305  0.001406
12    0.056779  0.057951  0.001172
13    0.046955  0.048253  0.001299
14    0.041104  0.044067  0.002964
15    0.039011  0.041305  0.002

### 6-2. get_feature_groups 누락 컬럼 확인

In [ ]:
# ------------------------------------------------------------------
# get_feature_groups가 놓치고 있는 컬럼이 있는지 확인
# ------------------------------------------------------------------
all_cols = set(train.columns)
excluded_known = {TARGET_COL, "seq"}  # target과 seq는 의도적으로 따로 처리하니 제외

groups = get_feature_groups(train.columns)
captured_cols = set()
for v in groups.values():
    captured_cols.update(v)

missing_cols = all_cols - captured_cols - excluded_known

print(f"전체 컬럼 수: {len(all_cols)}")
print(f"get_feature_groups가 잡아낸 컬럼 수: {len(captured_cols)}")
print(f"의도적으로 제외한 컬럼(target, seq): {len(excluded_known)}")
print(f"\n>>> 놓치고 있는 컬럼 수: {len(missing_cols)}")
print(f"놓친 컬럼 목록: {sorted(missing_cols)}")

# 놓친 컬럼들의 prefix 패턴 확인 (혹시 새로운 그룹이 있는지)
if missing_cols:
    prefixes = set()
    for col in missing_cols:
        # 마지막 언더스코어+숫자 부분 제거해서 prefix만 추출
        parts = col.rsplit("_", 1)
        prefix = parts[0] if len(parts) > 1 and parts[1].isdigit() else col
        prefixes.add(prefix)
    print(f"\n놓친 컬럼들의 prefix 패턴: {sorted(prefixes)}")

전체 컬럼 수: 131
get_feature_groups가 잡아낸 컬럼 수: 94
의도적으로 제외한 컬럼(target, seq): 2

>>> 놓치고 있는 컬럼 수: 36
놓친 컬럼 목록: ['history_b_1', 'history_b_10', 'history_b_11', 'history_b_12', 'history_b_13', 'history_b_14', 'history_b_15', 'history_b_16', 'history_b_17', 'history_b_18', 'history_b_19', 'history_b_2', 'history_b_20', 'history_b_21', 'history_b_22', 'history_b_23', 'history_b_24', 'history_b_25', 'history_b_26', 'history_b_27', 'history_b_28', 'history_b_29', 'history_b_3', 'history_b_30', 'history_b_4', 'history_b_5', 'history_b_6', 'history_b_7', 'history_b_8', 'history_b_9', 'hour_cos', 'hour_sin', 'seq_first', 'seq_last', 'seq_len', 'seq_len_log1p']

놓친 컬럼들의 prefix 패턴: ['history_b', 'hour_cos', 'hour_sin', 'seq_first', 'seq_last', 'seq_len', 'seq_len_log1p']


## 7. 실험

### (참고) 삭제된 이전 실험: 랜덤 split 기준 Optuna 튜닝 (2026-09-22)
중복 셀이라 삭제하고 결과만 기록:
- Best Score 0.35392 (30% 서브샘플, 30 trials)
- best params: learning_rate=0.0482, num_leaves=48, min_data_in_leaf=115, feature_fraction=0.7198, bagging_fraction=0.8441, lambda_l1=4.96e-05, lambda_l2=2.85e-04
- 재학습 Validation Score=0.35421 (AP=0.08190, WLL=0.59610) → `0922submission.csv`
- LB에서 기본값보다 낮게 나와서 아래 day7 기준 검증으로 전환


### 7-1. Target encoding (day7 기준 검증)

In [ ]:
train, test, te_cols = add_target_encoding(train, test, TARGET_ENCODE_COLS, target_col=TARGET_COL)
feature_cols_te = feature_cols + te_cols

model_day7_te = train_model_day7_val(train, feature_cols_te, cat_features)

/tmp/ipykernel_8074/642240807.py:53: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train[oof_col] = oof
/tmp/ipykernel_8074/642240807.py:57: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test[oof_col] = key_test.map(full_smooth).fillna(global_mean).astype(np.float32)


  te_inventory_id 추가 완료 (cardinality=18, range=[0.0067, 0.0632])


/tmp/ipykernel_8074/642240807.py:53: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train[oof_col] = oof
/tmp/ipykernel_8074/642240807.py:57: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test[oof_col] = key_test.map(full_smooth).fillna(global_mean).astype(np.float32)


  te_l_feat_14 추가 완료 (cardinality=3237, range=[0.0009, 0.0872])


/tmp/ipykernel_8074/642240807.py:53: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train[oof_col] = oof
/tmp/ipykernel_8074/642240807.py:57: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test[oof_col] = key_test.map(full_smooth).fillna(global_mean).astype(np.float32)


  te_inventory_id_x_hour 추가 완료 (cardinality=421, range=[0.0019, 0.0991])


/tmp/ipykernel_8074/642240807.py:53: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train[oof_col] = oof
/tmp/ipykernel_8074/642240807.py:57: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test[oof_col] = key_test.map(full_smooth).fillna(global_mean).astype(np.float32)


  te_inventory_id_x_day_of_week 추가 완료 (cardinality=125, range=[0.0020, 0.0811])

[4-D7] 모델 학습 (day_of_week==7 기준 validation)
  train(1~6요일): 9,177,844  /  valid(7요일): 1,526,335
  pos=177844, neg=9000000, scale_pos_weight=50.61
Training until validation scores don't improve for 50 rounds
[50]	valid_0's comp_score: 0.340303
[100]	valid_0's comp_score: 0.344656
[150]	valid_0's comp_score: 0.346149
[200]	valid_0's comp_score: 0.346889
[250]	valid_0's comp_score: 0.347193
[300]	valid_0's comp_score: 0.347431
[350]	valid_0's comp_score: 0.347586
[400]	valid_0's comp_score: 0.347656
[450]	valid_0's comp_score: 0.347735
[500]	valid_0's comp_score: 0.347819
[550]	valid_0's comp_score: 0.347975
[600]	valid_0's comp_score: 0.34807
[650]	valid_0's comp_score: 0.348132
[700]	valid_0's comp_score: 0.348142
Early stopping, best iteration is:
[667]	valid_0's comp_score: 0.348165

[day7 Validation] Score=0.34817  AP=0.07204  WLL=0.60181


In [ ]:
OUTPUT_PATH = os.path.join(DATA_DIR, "0923_1submission.csv")

predict_and_save(model_day7_te, test, feature_cols_te)  # 변수명은 실제 쓰신 것으로 맞춰주세요


[5] 예측 및 제출 파일 생성
  -> 제출 양식 컬럼: ['ID', 'clicked']
  -> /content/drive/MyDrive/ESAA/0923_1submission.csv 저장 완료 (shape=(1527298, 2))


제출: 09230147

### 7-2. 기본 파라미터 (day7 기준 검증)

In [ ]:
model_day7_default = train_model_day7_val(train, feature_cols, cat_features)


[4-D7] 모델 학습 (day_of_week==7 기준 validation)
  train(1~6요일): 9,177,844  /  valid(7요일): 1,526,335
  pos=177844, neg=9000000, scale_pos_weight=50.61
Training until validation scores don't improve for 50 rounds
[50]	valid_0's comp_score: 0.340165
[100]	valid_0's comp_score: 0.344542
[150]	valid_0's comp_score: 0.346385
[200]	valid_0's comp_score: 0.347078
[250]	valid_0's comp_score: 0.347439
[300]	valid_0's comp_score: 0.347675
[350]	valid_0's comp_score: 0.347869
[400]	valid_0's comp_score: 0.347998
[450]	valid_0's comp_score: 0.348153
[500]	valid_0's comp_score: 0.34822
[550]	valid_0's comp_score: 0.348247
[600]	valid_0's comp_score: 0.348325
[650]	valid_0's comp_score: 0.34834
[700]	valid_0's comp_score: 0.348346
[750]	valid_0's comp_score: 0.348427
[800]	valid_0's comp_score: 0.348437
[850]	valid_0's comp_score: 0.348493
[900]	valid_0's comp_score: 0.348478
Early stopping, best iteration is:
[882]	valid_0's comp_score: 0.348502

[day7 Validation] Score=0.34850  AP=0.07295  WLL=0.60243

In [ ]:
OUTPUT_PATH = os.path.join(DATA_DIR, "0923_0220submission.csv")

predict_and_save(model_day7_default, test, feature_cols)


[5] 예측 및 제출 파일 생성
  -> 제출 양식 컬럼: ['ID', 'clicked']
  -> /content/drive/MyDrive/ESAA/0923_0220submission.csv 저장 완료 (shape=(1527298, 2))


제출: 09230224

### 7-3. Optuna 튜닝 (day7 기준)

In [ ]:
study = run_tuning(train, feature_cols, cat_features)

튜닝용 데이터: 전체의 30%로 30회 trial 진행


[I 2026-09-22 17:24:08,761] A new study created in memory with name: no-name-aea390af-2d61-461e-abb3-c601bba6be32


  튜닝 데이터 (서브샘플 30%): train(1~6요일)=2,753,354  valid(7요일)=457,900


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-09-22 17:25:55,678] Trial 0 finished with value: 0.34204528121493477 and parameters: {'learning_rate': 0.08499731865368554, 'num_leaves': 54, 'min_data_in_leaf': 87, 'feature_fraction': 0.977170202320692, 'bagging_fraction': 0.8287609908806908, 'lambda_l1': 5.054736815920442e-05, 'lambda_l2': 0.4389306385050914}. Best is trial 0 with value: 0.34204528121493477.
[I 2026-09-22 17:27:44,702] Trial 1 finished with value: 0.3414499493536762 and parameters: {'learning_rate': 0.07137418512286794, 'num_leaves': 95, 'min_data_in_leaf': 168, 'feature_fraction': 0.804163984161088, 'bagging_fraction': 0.6863497047712301, 'lambda_l1': 0.0003111993126138757, 'lambda_l2': 3.119499862595168e-07}. Best is trial 0 with value: 0.34204528121493477.
[I 2026-09-22 17:36:39,627] Trial 2 finished with value: 0.3429597106200869 and parameters: {'learning_rate': 0.013999518911522593, 'num_leaves': 255, 'min_data_in_leaf': 130, 'feature_fraction': 0.813917835855737, 'bagging_fraction': 0.8717396419849262

In [ ]:
model = apply_best_params_and_retrain(study, train, feature_cols, cat_features)

튜닝된 파라미터로 학습: {'num_leaves': 84, 'learning_rate': 0.01499044526807465, 'min_data_in_leaf': 83, 'feature_fraction': 0.8241113845518219, 'bagging_fraction': 0.7186588651230175, 'lambda_l1': 4.061223972890259, 'lambda_l2': 4.409517509864513}

[4-D7] 모델 학습 (day_of_week==7 기준 validation)
  실제 사용 파라미터: {'num_leaves': 84, 'learning_rate': 0.01499044526807465, 'min_data_in_leaf': 83, 'feature_fraction': 0.8241113845518219, 'bagging_fraction': 0.7186588651230175, 'lambda_l1': 4.061223972890259, 'lambda_l2': 4.409517509864513}
  num_boost_round=2000
  train(1~6요일): 9,177,844  /  valid(7요일): 1,526,335
  pos=177844, neg=9000000, scale_pos_weight=50.61
Training until validation scores don't improve for 50 rounds
[50]	valid_0's comp_score: 0.316071
[100]	valid_0's comp_score: 0.334657
[150]	valid_0's comp_score: 0.339913
[200]	valid_0's comp_score: 0.341987
[250]	valid_0's comp_score: 0.343411
[300]	valid_0's comp_score: 0.344461
[350]	valid_0's comp_score: 0.345329
[400]	valid_0's comp_score: 0.346

In [ ]:
OUTPUT_PATH = os.path.join(DATA_DIR, "0923_0437_optuna_day7submission.csv")

predict_and_save(model, test, feature_cols)


[5] 예측 및 제출 파일 생성
  -> 제출 양식 컬럼: ['ID', 'clicked']
  -> /content/drive/MyDrive/ESAA/0923_0437_optuna_day7submission.csv 저장 완료 (shape=(1527298, 2))


### 7-4. 튜닝 파라미터: seed 앙상블 / 전체 데이터 재학습 / 기본값 모델과 블렌딩

In [ ]:
lgb_params = {
    "num_leaves": 84,
    "learning_rate": 0.01499044526807465,
    "min_data_in_leaf": 83,
    "feature_fraction": 0.8241113845518219,
    "bagging_fraction": 0.7186588651230175,
    "lambda_l1": 4.061223972890259,
    "lambda_l2": 4.409517509864513,
}

In [ ]:
models = train_seed_ensemble(train, feature_cols, cat_features, lgb_params, n_seeds=5)
predict_and_save_seed_ensemble(models, test, feature_cols)


############################################################
# Seed 100 (1/5)
############################################################

[4-D7] 모델 학습 (day_of_week==7 기준 validation)
  실제 사용 파라미터: {'num_leaves': 84, 'learning_rate': 0.01499044526807465, 'min_data_in_leaf': 83, 'feature_fraction': 0.8241113845518219, 'bagging_fraction': 0.7186588651230175, 'lambda_l1': 4.061223972890259, 'lambda_l2': 4.409517509864513}
  num_boost_round=2000, seed=100
  train(1~6요일): 9,177,844  /  valid(7요일): 1,526,335
  pos=177844, neg=9000000, scale_pos_weight=50.61
Training until validation scores don't improve for 50 rounds
[50]	valid_0's comp_score: 0.316131
[100]	valid_0's comp_score: 0.33465
[150]	valid_0's comp_score: 0.339871
[200]	valid_0's comp_score: 0.342056
[250]	valid_0's comp_score: 0.343434
[300]	valid_0's comp_score: 0.344329
[350]	valid_0's comp_score: 0.345312
[400]	valid_0's comp_score: 0.345966
[450]	valid_0's comp_score: 0.346389
[500]	valid_0's comp_score: 0.346812
[550]	valid_

In [ ]:
final_model = train_final_full_data(train, feature_cols, cat_features, lgb_params, num_boost_round=1982)
predict_and_save(final_model, test, feature_cols)


[FINAL] 전체 데이터(day7 포함)로 최종 학습, num_boost_round=1982
  전체 pos=204179, neg=10500000, scale_pos_weight=51.43, seed=42

[5] 예측 및 제출 파일 생성
  -> 제출 양식 컬럼: ['ID', 'clicked']
  -> /content/drive/MyDrive/ESAA/submission.csv 저장 완료 (shape=(1527298, 2))


In [ ]:
# ------------------------------------------------------------------
# 0914 스타일(기본 파라미터) 모델을 이 세션에서 재학습 -> day7 튜닝 모델과 블렌딩
# ------------------------------------------------------------------
default_params = {
    "num_leaves": 63,
    "learning_rate": 0.05,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "lambda_l1": 0.0,
    "lambda_l2": 0.0,
}
model_default = train_final_full_data(train, feature_cols, cat_features, default_params, num_boost_round=2000)

# 주의: train_final_full_data로 만든 모델은 valid set이 없어서 best_iteration이 없음
# -> num_iteration 생략해서 전체 트리 다 쓰도록 함 (0914 원본이랑 동일한 로직)
pred_tuned = final_model.predict(test[feature_cols])
pred_default = model_default.predict(test[feature_cols])

blend_pred = 0.5 * pred_tuned + 0.5 * pred_default

# 제출 파일 저장 (predict_and_save 로직 재사용, 파일명만 새로)
BLEND_OUTPUT_PATH = os.path.join(DATA_DIR, "0923_blend_submission.csv")

if os.path.exists(SAMPLE_SUBMISSION_PATH):
    sample_sub = pd.read_csv(SAMPLE_SUBMISSION_PATH)
    pred_map = pd.Series(blend_pred, index=test[ID_COL].values)
    target_col_in_sample = [c for c in sample_sub.columns if c != ID_COL][0]
    submission = sample_sub.copy()
    submission[target_col_in_sample] = submission[ID_COL].map(pred_map)
    assert submission[target_col_in_sample].isna().sum() == 0
else:
    submission = pd.DataFrame({ID_COL: test[ID_COL], TARGET_COL: blend_pred})

submission.to_csv(BLEND_OUTPUT_PATH, index=False)
print(f"블렌딩 제출 파일 저장 완료: {BLEND_OUTPUT_PATH} (shape={submission.shape})")


[FINAL] 전체 데이터(day7 포함)로 최종 학습, num_boost_round=2000
  전체 pos=204179, neg=10500000, scale_pos_weight=51.43, seed=42
블렌딩 제출 파일 저장 완료: /content/drive/MyDrive/ESAA/0923_blend_submission.csv (shape=(1527298, 2))


### 7-5. history_b 피처 추가

In [ ]:
history_b_cols = [c for c in train.columns if c.startswith("history_b_")]
feature_cols = feature_cols + history_b_cols
print(f"추가된 컬럼 수: {len(history_b_cols)}")
print(f"총 피처 수: {len(feature_cols)}")

추가된 컬럼 수: 30
총 피처 수: 129


In [ ]:
model_with_historyb = train_model_day7_val(train, feature_cols, cat_features, lgb_params=lgb_params)


[4-D7] 모델 학습 (day_of_week==7 기준 validation)
  실제 사용 파라미터: {'num_leaves': 84, 'learning_rate': 0.01499044526807465, 'min_data_in_leaf': 83, 'feature_fraction': 0.8241113845518219, 'bagging_fraction': 0.7186588651230175, 'lambda_l1': 4.061223972890259, 'lambda_l2': 4.409517509864513}
  num_boost_round=2000, seed=42
  train(1~6요일): 9,177,844  /  valid(7요일): 1,526,335
  pos=177844, neg=9000000, scale_pos_weight=50.61
Training until validation scores don't improve for 50 rounds
[50]	valid_0's comp_score: 0.316194
[100]	valid_0's comp_score: 0.334597
[150]	valid_0's comp_score: 0.339835
[200]	valid_0's comp_score: 0.34205
[250]	valid_0's comp_score: 0.34336
[300]	valid_0's comp_score: 0.344435
[350]	valid_0's comp_score: 0.345359
[400]	valid_0's comp_score: 0.346077
[450]	valid_0's comp_score: 0.346587
[500]	valid_0's comp_score: 0.346979
[550]	valid_0's comp_score: 0.347219
[600]	valid_0's comp_score: 0.347416
[650]	valid_0's comp_score: 0.34753
[700]	valid_0's comp_score: 0.347685
[750]	v

In [ ]:
imp = pd.DataFrame({
    "feature": model_with_historyb.feature_name(),
    "importance": model_with_historyb.feature_importance(importance_type="gain"),
}).sort_values("importance", ascending=False)

print("전체 Top 20:")
print(imp.head(20).to_string(index=False))

print("\nhistory_b만 따로 보기:")
print(imp[imp["feature"].str.startswith("history_b_")].head(10).to_string(index=False))

전체 Top 20:
                      feature   importance
                  history_a_1 2.487179e+07
                 inventory_id 2.059447e+07
history_a_1_minus_history_a_2 8.933171e+06
                         hour 7.390339e+06
                     feat_e_3 2.482820e+06
                    age_group 2.064693e+06
                     l_feat_6 1.602114e+06
                     feat_d_4 1.545110e+06
                     l_feat_5 1.409060e+06
                     feat_c_8 1.401911e+06
                     l_feat_7 1.284513e+06
                    l_feat_10 1.271739e+06
                     l_feat_2 1.250872e+06
                    l_feat_14 1.154290e+06
                     feat_b_6 1.027720e+06
                     l_feat_9 1.014951e+06
                     feat_b_5 9.681888e+05
                    l_feat_15 9.617811e+05
                     feat_b_3 9.156246e+05
                    l_feat_12 8.672341e+05

history_b만 따로 보기:
     feature    importance
history_b_30 800132.750183
 history_b_3 

### 7-6. 공개공유코드 DNN (Wide&Deep + Cross + BiLSTM) — day7 검증 버전
`0923_공개공유코드.ipynb`의 모델 구조를 가져오되, 아래를 고쳐서 LightGBM과 같은 조건(1~6요일 학습 / day7 검증 / 대회 산식)으로 비교:
- 검증 없음 → day7 검증 + epoch마다 `competition_score` 측정, best epoch 저장 (early stopping)
- 스케줄러가 배치마다 `step()`되어 2·4·8 배치마다 LR 재시작 → epoch 단위 CosineAnnealing
- seq 값을 float로 LSTM에 넣음 → 토큰 ID로 보고 `nn.Embedding` + 최근 `SEQ_MAX_LEN`개만 사용
- 행 단위 `__getitem__` + `np.fromstring` (느림) → 미리 배열로 파싱 후 텐서 인덱싱으로 배치 구성
- 수치형은 train 기준 표준화 + 결측 0 (공개 코드는 결측 0만)
- 범주형 인코딩은 train 기준 사전, test에만 있는 값은 0(UNK)

결과물 `dnn_val_pred`(day7 예측), `dnn_test_pred`(test 예측)는 `.npy`로도 저장 → 7-7 블렌딩에서 사용.
GPU 런타임 권장 (CPU면 `DNN_CFG["TRAIN_SAMPLE_FRAC"]`를 낮출 것).


In [ ]:
import time
import torch
import torch.nn as nn

DNN_CFG = {
    "TRAIN_SAMPLE_FRAC": 1.0,   # 1~6요일 학습 데이터 중 사용 비율 (메모리/시간 보고 조절, 1.0 = 전체)
    "SEQ_MAX_LEN": 50,          # seq에서 최근 N개 토큰만 사용
    "SEQ_HASH_BUCKETS": 2000,     # 토큰 ID 버킷 수 (seq 토큰 최대값 ~584 확인 -> 충돌 없음)
    "SEQ_EMB_DIM": 16,
    "CAT_EMB_DIM": 16,
    "LSTM_HIDDEN": 64,
    "HIDDEN_UNITS": [512, 256, 128],
    "DROPOUT": [0.1, 0.2, 0.3],
    "BATCH_SIZE": 4096,
    "EPOCHS": 6,
    "PATIENCE": 2,              # day7 Score가 이 epoch 수만큼 안 오르면 중단
    "LR": 1e-3,
    "WEIGHT_DECAY": 1e-5,
    "SEED": SEED,
}
DNN_CAT_COLS = ["gender", "age_group", "inventory_id", "day_of_week", "hour", "l_feat_14"]  # l_feat_14는 공개 코드처럼 임베딩
DNN_VAL_PRED_PATH = os.path.join(DATA_DIR, "0923_dnn_day7_val_pred.npy")
DNN_TEST_PRED_PATH = os.path.join(DATA_DIR, "0923_dnn_test_pred.npy")


def seed_everything(seed):
    import random
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(DNN_CFG["SEED"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cpu":
    print("  !! GPU 없음 -> 매우 느림. 런타임 유형을 GPU로 바꾸거나 TRAIN_SAMPLE_FRAC를 낮출 것")

Device: cuda


In [ ]:
# ------------------------------------------------------------------
# DNN 입력 준비: train/test 원본 DataFrame은 건드리지 않고 numpy 배열만 새로 만듦
# ------------------------------------------------------------------
def parse_seq_tail(seq_series: pd.Series, max_len, n_buckets):
    """콤마 구분 seq 문자열에서 최근 max_len개 토큰만 파싱 -> [N, max_len] int32 (앞쪽 정렬, 0=패딩)."""
    n = len(seq_series)
    out = np.zeros((n, max_len), dtype=np.int32)
    lens = np.ones(n, dtype=np.int64)  # 빈 seq도 길이 1(패딩 토큰)로 처리 -> pack_padded 에러 방지
    for i, s in enumerate(seq_series.values):
        if not isinstance(s, str) or not s:
            continue
        tail = s.rsplit(",", max_len)[-max_len:]  # 뒤에서부터만 잘라서 긴 seq도 빠르게
        toks = np.array([t for t in tail if t], dtype=np.int64)
        if len(toks) == 0:
            continue
        out[i, :len(toks)] = (toks % n_buckets) + 1  # 0은 패딩용으로 비워둠
        lens[i] = len(toks)
    return out, lens


def build_cat_codes(fit_series: pd.Series, series_list):
    """fit_series 기준 사전으로 코드화. 0 = 결측/처음 보는 값, 1.. = 실제 카테고리."""
    cats = pd.Index(pd.unique(np.asarray(fit_series.dropna())))
    codes = [pd.Categorical(np.asarray(s), categories=cats).codes.astype(np.int64) + 1 for s in series_list]
    return codes, len(cats) + 1


def prepare_dnn_inputs(train_df, test_df, feature_cols, cfg):
    t0 = time.time()
    val_mask = (train_df["day_of_week"].astype(str) == "7").values
    tr_idx = np.where(~val_mask)[0]
    va_idx = np.where(val_mask)[0]
    if cfg["TRAIN_SAMPLE_FRAC"] < 1.0:
        rng = np.random.RandomState(cfg["SEED"])
        tr_idx = np.sort(rng.choice(tr_idx, int(len(tr_idx) * cfg["TRAIN_SAMPLE_FRAC"]), replace=False))
    print(f"  DNN train(1~6요일, {cfg['TRAIN_SAMPLE_FRAC']*100:.0f}%): {len(tr_idx):,}  /  valid(7요일): {len(va_idx):,}  /  test: {len(test_df):,}")

    tr = train_df.iloc[tr_idx]
    va = train_df.iloc[va_idx]

    cat_cols = [c for c in DNN_CAT_COLS if c in train_df.columns]
    num_cols = [c for c in feature_cols if c not in cat_cols]

    # 수치형: train 기준 평균/표준편차로 표준화, 결측 0, 극단값 클리핑
    def to_num(df):
        return df[num_cols].to_numpy(dtype=np.float32, na_value=np.nan)
    X_tr = to_num(tr)
    mean = np.nanmean(X_tr, axis=0)
    std = np.nanstd(X_tr, axis=0)
    mean = np.nan_to_num(mean, nan=0.0)
    std = np.where(np.isfinite(std) & (std > 1e-6), std, 1.0)

    def scale(X):
        X = (X - mean) / std
        X = np.nan_to_num(X, nan=0.0, posinf=10.0, neginf=-10.0)
        return np.clip(X, -10, 10).astype(np.float32)
    num = {"train": scale(X_tr), "valid": scale(to_num(va)), "test": scale(to_num(test_df))}
    del X_tr

    # 범주형: train 사전 기준 코드화
    cat = {"train": [], "valid": [], "test": []}
    cardinalities = []
    for c in cat_cols:
        (c_tr, c_va, c_te), card = build_cat_codes(tr[c], [tr[c], va[c], test_df[c]])
        cat["train"].append(c_tr); cat["valid"].append(c_va); cat["test"].append(c_te)
        cardinalities.append(card)
    cat = {k: np.stack(v, axis=1) for k, v in cat.items()}

    # seq: 최근 SEQ_MAX_LEN개 토큰
    seq = {}
    for name, df in [("train", tr), ("valid", va), ("test", test_df)]:
        ts = time.time()
        seq[name] = parse_seq_tail(df["seq"], cfg["SEQ_MAX_LEN"], cfg["SEQ_HASH_BUCKETS"])
        print(f"  seq 파싱 {name}: {time.time()-ts:.0f}s")

    y = {
        "train": tr[TARGET_COL].to_numpy(dtype=np.float32),
        "valid": va[TARGET_COL].to_numpy(dtype=np.float32),
    }
    print(f"  수치형 {len(num_cols)}개 / 범주형 {len(cat_cols)}개 {dict(zip(cat_cols, cardinalities))}")
    print(f"  준비 완료 ({time.time()-t0:.0f}s)")
    return {"num": num, "cat": cat, "seq": seq, "y": y,
            "num_cols": num_cols, "cat_cols": cat_cols, "cardinalities": cardinalities}


#dnn_data = prepare_dnn_inputs(train, test, feature_cols, DNN_CFG)
#gc.collect()

In [ ]:
# ------------------------------------------------------------------
# 모델: 공개 코드 구조 유지 (BN 수치형 + 범주형 임베딩 + BiLSTM(seq) -> Cross Network -> MLP)
# 차이점: seq 토큰을 float가 아니라 Embedding으로 넣음
# ------------------------------------------------------------------
class CrossNetwork(nn.Module):
    def __init__(self, input_dim, num_layers=2):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(input_dim, 1, bias=True) for _ in range(num_layers)])

    def forward(self, x0):
        x = x0
        for w in self.layers:
            x = x0 * w(x) + x
        return x


class WideDeepCTR(nn.Module):
    def __init__(self, num_features, cat_cardinalities, cfg):
        super().__init__()
        self.emb_layers = nn.ModuleList([nn.Embedding(card, cfg["CAT_EMB_DIM"]) for card in cat_cardinalities])
        cat_input_dim = cfg["CAT_EMB_DIM"] * len(cat_cardinalities)
        self.bn_num = nn.BatchNorm1d(num_features)
        self.seq_emb = nn.Embedding(cfg["SEQ_HASH_BUCKETS"] + 1, cfg["SEQ_EMB_DIM"], padding_idx=0)
        self.lstm = nn.LSTM(input_size=cfg["SEQ_EMB_DIM"], hidden_size=cfg["LSTM_HIDDEN"],
                            num_layers=2, batch_first=True, bidirectional=True)
        input_dim = num_features + cat_input_dim + cfg["LSTM_HIDDEN"] * 2
        self.cross = CrossNetwork(input_dim, num_layers=2)
        layers = []
        for i, h in enumerate(cfg["HIDDEN_UNITS"]):
            layers += [nn.Linear(input_dim, h), nn.ReLU(), nn.Dropout(cfg["DROPOUT"][i % len(cfg["DROPOUT"])])]
            input_dim = h
        layers += [nn.Linear(input_dim, 1)]
        self.mlp = nn.Sequential(*layers)

    def forward(self, num_x, cat_x, seqs, seq_lengths):
        num_x = self.bn_num(num_x)
        cat_feat = torch.cat([emb(cat_x[:, i]) for i, emb in enumerate(self.emb_layers)], dim=1)
        packed = nn.utils.rnn.pack_padded_sequence(self.seq_emb(seqs), seq_lengths.cpu(),
                                                   batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        h = torch.cat([h_n[-2], h_n[-1]], dim=1)
        z = torch.cat([num_x, cat_feat, h], dim=1)
        return self.mlp(self.cross(z)).squeeze(1)


def iter_batches(data, split, batch_size, shuffle, rng=None):
    n = len(data["num"][split])
    order = rng.permutation(n) if shuffle else np.arange(n)
    seq_arr, seq_len = data["seq"][split]
    for s in range(0, n, batch_size):
        idx = order[s:s + batch_size]
        lens = seq_len[idx]
        max_len = int(lens.max())  # 배치 내 최대 길이까지만 잘라서 LSTM 연산량 절약
        batch = [
            torch.from_numpy(data["num"][split][idx]),
            torch.from_numpy(data["cat"][split][idx]),
            torch.from_numpy(seq_arr[idx, :max_len].astype(np.int64)),
            torch.from_numpy(lens),
        ]
        if split in data["y"]:
            batch.append(torch.from_numpy(data["y"][split][idx]))
        yield batch


@torch.no_grad()
def predict_dnn(model, data, split, batch_size):
    model.eval()
    outs = []
    for num_x, cat_x, seqs, lens, *_ in iter_batches(data, split, batch_size, shuffle=False):
        logits = model(num_x.to(device), cat_x.to(device), seqs.to(device), lens)
        outs.append(torch.sigmoid(logits).float().cpu().numpy())
    return np.concatenate(outs)


def train_dnn_day7(data, cfg):
    print("\n" + "=" * 60)
    print("[4-DNN] DNN 학습 (day_of_week==7 기준 validation)")
    print("=" * 60)
    model = WideDeepCTR(len(data["num_cols"]), data["cardinalities"], cfg).to(device)

    y_tr = data["y"]["train"]
    pos_weight_value = (len(y_tr) - y_tr.sum()) / max(y_tr.sum(), 1)  # LightGBM scale_pos_weight와 같은 발상
    print(f"  pos_weight={pos_weight_value:.2f}")
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_value], device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["LR"], weight_decay=cfg["WEIGHT_DECAY"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["EPOCHS"])  # epoch 단위로 step

    rng = np.random.RandomState(cfg["SEED"])
    best_score, best_state, best_epoch, bad = -1.0, None, 0, 0
    n_batches = int(np.ceil(len(y_tr) / cfg["BATCH_SIZE"]))

    for epoch in range(1, cfg["EPOCHS"] + 1):
        model.train()
        t0, total_loss = time.time(), 0.0
        for b, (num_x, cat_x, seqs, lens, ys) in enumerate(iter_batches(data, "train", cfg["BATCH_SIZE"], True, rng), 1):
            num_x, cat_x, seqs, ys = num_x.to(device), cat_x.to(device), seqs.to(device), ys.to(device)
            optimizer.zero_grad()
            loss = criterion(model(num_x, cat_x, seqs, lens), ys)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(ys)
            if b % 200 == 0:
                print(f"    batch {b}/{n_batches}  loss={loss.item():.4f}  ({time.time()-t0:.0f}s)")
        scheduler.step()

        val_pred = predict_dnn(model, data, "valid", cfg["BATCH_SIZE"] * 4)
        score, ap, wll = competition_score(data["y"]["valid"], val_pred)
        print(f"[Epoch {epoch}] train_loss={total_loss/len(y_tr):.4f}  "
              f"day7 Score={score:.5f}  AP={ap:.5f}  WLL={wll:.5f}  ({time.time()-t0:.0f}s)")

        if score > best_score:
            best_score, best_epoch, bad = score, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= cfg["PATIENCE"]:
                print(f"  early stopping (best epoch={best_epoch})")
                break

    model.load_state_dict(best_state)
    print(f"\n[DNN day7 Validation] best epoch={best_epoch}  Score={best_score:.5f}")
    return model

In [ ]:
dnn_model = train_dnn_day7(dnn_data, DNN_CFG)

dnn_val_pred = predict_dnn(dnn_model, dnn_data, "valid", DNN_CFG["BATCH_SIZE"] * 4)
dnn_test_pred = predict_dnn(dnn_model, dnn_data, "test", DNN_CFG["BATCH_SIZE"] * 4)
score, ap, wll = competition_score(dnn_data["y"]["valid"], dnn_val_pred)
print(f"[DNN day7 Validation] Score={score:.5f}  AP={ap:.5f}  WLL={wll:.5f}")
print("  비교: LightGBM seed 앙상블 0.34882 / 기본 파라미터 0.34850")

# 블렌딩용으로 저장 (런타임 끊겨도 다시 학습 안 해도 되게)
np.save(DNN_VAL_PRED_PATH, dnn_val_pred)
np.save(DNN_TEST_PRED_PATH, dnn_test_pred)
print(f"  -> {DNN_VAL_PRED_PATH}, {DNN_TEST_PRED_PATH} 저장 완료")


[4-DNN] DNN 학습 (day_of_week==7 기준 validation)
  pos_weight=50.61
    batch 200/2241  loss=1.1997  (14s)
    batch 400/2241  loss=1.2609  (27s)
    batch 600/2241  loss=1.3324  (39s)
    batch 800/2241  loss=1.0805  (52s)
    batch 1000/2241  loss=1.2242  (64s)
    batch 1200/2241  loss=1.3179  (77s)
    batch 1400/2241  loss=1.2235  (89s)
    batch 1600/2241  loss=1.2401  (102s)
    batch 1800/2241  loss=1.1116  (114s)
    batch 2000/2241  loss=1.1334  (127s)
    batch 2200/2241  loss=1.2881  (139s)
[Epoch 1] train_loss=1.2039  day7 Score=0.34361  AP=0.06529  WLL=0.60789  (149s)
    batch 200/2241  loss=1.1552  (13s)
    batch 400/2241  loss=1.1151  (25s)
    batch 600/2241  loss=1.2427  (38s)
    batch 800/2241  loss=1.1345  (50s)
    batch 1000/2241  loss=1.1838  (63s)
    batch 1200/2241  loss=1.2391  (75s)
    batch 1400/2241  loss=1.2291  (88s)
    batch 1600/2241  loss=1.0724  (100s)
    batch 1800/2241  loss=1.2451  (113s)
    batch 2000/2241  loss=1.2331  (125s)
    batch 2200

In [ ]:
# (선택) DNN 단독 제출 파일 - test 행 순서 그대로 ID와 매핑
OUTPUT_PATH = os.path.join(DATA_DIR, "0923_dnn_day7_submission.csv")
sample_sub = pd.read_csv(SAMPLE_SUBMISSION_PATH)
pred_map = pd.Series(dnn_test_pred, index=test[ID_COL].values)
sample_sub["clicked"] = sample_sub[ID_COL].map(pred_map)
assert sample_sub["clicked"].isna().sum() == 0
sample_sub.to_csv(OUTPUT_PATH, index=False)
print(f"  -> {OUTPUT_PATH} 저장 완료 (shape={sample_sub.shape})")

  -> /content/drive/MyDrive/ESAA/0923_dnn_day7_submission.csv 저장 완료 (shape=(1527298, 2))


### 7-7. LightGBM + DNN 블렌딩 (가중치는 day7 검증, test 예측은 전체 데이터 재학습 모델)
- **가중치 탐색**: 1~6요일로 학습한 LightGBM / DNN의 day7 예측으로 대회 산식이 가장 높은 DNN 비중을 찾음 (prob 평균 vs logit 평균 비교)
- **test 예측**: 두 모델 모두 day7 포함 **전체 train으로 재학습**한 모델 사용 (day7을 빼고 학습하면 LB에서 손해였음 - 2026-09-23 확인)
  - LightGBM: day7 검증 best_iteration 그대로 고정 라운드로 재학습 (`train_final_full_data`)
  - DNN: 7-6과 같은 설정/LR 스케줄로 best epoch까지만 학습 (검증 없이 고정 epoch)
- 예측은 전부 `.npy`로 저장 -> 런타임 끊겨도 재학습 없이 블렌딩만 다시 가능
- 실행 전제: 1장, 2장(day7/전체 학습 함수), 5장, (7-5 history_b 셀), 7-6 전체

 참고할 점
  - 예측값은 전부 .npy로
    저장돼요. 파라미터, 피처,
    DNN 설정을 바꾼 뒤 다시 돌릴
    때는
    0923_lgb_day7_val_pred.npy,
    0923_lgb_full_test_pred.npy,
    0923_dnn_full_test_pred.npy
    를 지워야 해요. 안 지우면
    예전 예측을 그대로
    재사용해요.
  - day7 모델과 전체 재학습
    모델은 서로 다른 모델이라,
    day7에서 찾은 가중치가 전체
    재학습 모델에 정확히
    맞는다는 보장은 없어요.
    7-7-3에서 test 평균 예측을
    출력하니까, day7 평균과 크게
    다르면 한 번 확인해 보세요.

In [ ]:
# ------------------------------------------------------------------
# 7-7-1. LightGBM: day7 검증 예측(가중치 탐색용) + 전체 데이터 재학습 test 예측(제출용)
# ------------------------------------------------------------------
# 파라미터나 feature_cols를 바꾸면 아래 .npy 두 개를 지우고 다시 실행할 것 (안 지우면 예전 예측을 재사용함)
LGB_VAL_PRED_PATH = os.path.join(DATA_DIR, "0923_lgb_day7_val_pred.npy")
LGB_FULL_TEST_PRED_PATH = os.path.join(DATA_DIR, "0923_lgb_full_test_pred.npy")

# 7-4 튜닝 파라미터 (7-5 history_b 포함 day7 0.34875, best_iteration=1672)
BLEND_LGB_PARAMS = {
    "num_leaves": 84,
    "learning_rate": 0.01499044526807465,
    "min_data_in_leaf": 83,
    "feature_fraction": 0.8241113845518219,
    "bagging_fraction": 0.7186588651230175,
    "lambda_l1": 4.061223972890259,
    "lambda_l2": 4.409517509864513,
}

blend_val_mask = (train["day_of_week"].astype(str) == "7").values
blend_y_valid = train.loc[blend_val_mask, TARGET_COL].to_numpy(dtype=np.int8)
print(f"feature_cols {len(feature_cols)}개 사용 (7-5 history_b 포함 여부 확인)")

if os.path.exists(LGB_VAL_PRED_PATH) and os.path.exists(LGB_FULL_TEST_PRED_PATH):
    lgb_val_pred = np.load(LGB_VAL_PRED_PATH)
    lgb_full_test_pred = np.load(LGB_FULL_TEST_PRED_PATH)
    print("LightGBM 예측은 저장된 .npy에서 불러옴")
else:
    # 7-5에서 같은 피처로 학습한 모델이 메모리에 있으면 재사용 (day7 학습 한 번 절약)
    if "model_with_historyb" in globals() and model_with_historyb.feature_name() == list(feature_cols):
        lgb_day7_model = model_with_historyb
        print("7-5 model_with_historyb 재사용")
    else:
        lgb_day7_model = train_model_day7_val(train, feature_cols, cat_features, lgb_params=BLEND_LGB_PARAMS)
    lgb_best_iter = lgb_day7_model.best_iteration
    lgb_val_pred = lgb_day7_model.predict(train.loc[blend_val_mask, feature_cols], num_iteration=lgb_best_iter)
    np.save(LGB_VAL_PRED_PATH, lgb_val_pred)

    # 전체 데이터(day7 포함) 재학습 -> 제출용 test 예측
    lgb_full_model = train_final_full_data(train, feature_cols, cat_features, BLEND_LGB_PARAMS,
                                           num_boost_round=lgb_best_iter)
    lgb_full_test_pred = lgb_full_model.predict(test[feature_cols])  # valid 없이 학습 -> 전체 트리 사용
    np.save(LGB_FULL_TEST_PRED_PATH, lgb_full_test_pred)
    print(f"  -> {LGB_VAL_PRED_PATH}, {LGB_FULL_TEST_PRED_PATH} 저장 완료 (best_iteration={lgb_best_iter})")
    gc.collect()

feature_cols 129개 사용 (7-5 history_b 포함 여부 확인)
LightGBM 예측은 저장된 .npy에서 불러옴


In [ ]:
# ------------------------------------------------------------------
# 7-7-2. DNN: 전체 데이터(day7 포함)로 재학습 -> 제출용 test 예측
# (day7 검증 예측은 7-6의 dnn_val_pred를 그대로 가중치 탐색에 사용)
# ------------------------------------------------------------------
DNN_FULL_TEST_PRED_PATH = os.path.join(DATA_DIR, "0923_dnn_full_test_pred.npy")
DNN_FULL_EPOCHS = 5  # 7-6 로그 "best epoch=5" (TRAIN_SAMPLE_FRAC=1.0) (7-6 설정을 바꿔 다시 돌리면 여기도 갱신 + 위 .npy 삭제)


def prepare_dnn_inputs_full(train_df, test_df, feature_cols, cfg):
    """7-6 prepare_dnn_inputs의 전체 데이터 버전: 검증 split 없이 day7 포함 전체 train으로 표준화/사전 생성."""
    t0 = time.time()
    tr_idx = np.arange(len(train_df))
    if cfg["TRAIN_SAMPLE_FRAC"] < 1.0:
        rng = np.random.RandomState(cfg["SEED"])
        tr_idx = np.sort(rng.choice(tr_idx, int(len(tr_idx) * cfg["TRAIN_SAMPLE_FRAC"]), replace=False))
    print(f"  DNN train(전체 1~7요일, {cfg['TRAIN_SAMPLE_FRAC']*100:.0f}%): {len(tr_idx):,}  /  test: {len(test_df):,}")
    # 100%면 복사 없이 원본 사용 (iloc 복사본이 수 GB)
    tr = train_df if len(tr_idx) == len(train_df) else train_df.iloc[tr_idx]

    cat_cols = [c for c in DNN_CAT_COLS if c in train_df.columns]
    num_cols = [c for c in feature_cols if c not in cat_cols]

    X_tr = tr[num_cols].to_numpy(dtype=np.float32, na_value=np.nan)
    mean = np.nan_to_num(np.nanmean(X_tr, axis=0), nan=0.0).astype(np.float32)
    std = np.nanstd(X_tr, axis=0)
    std = np.where(np.isfinite(std) & (std > 1e-6), std, 1.0).astype(np.float32)

    def scale(X):
        # 제자리 연산: 임시 복사본 없이 X 배열 하나만 사용 (전체 데이터에서 OOM 방지)
        X -= mean
        X /= std
        np.nan_to_num(X, copy=False, nan=0.0, posinf=10.0, neginf=-10.0)
        np.clip(X, -10, 10, out=X)
        return X
    num = {"train": scale(X_tr), "test": scale(test_df[num_cols].to_numpy(dtype=np.float32, na_value=np.nan))}
    del X_tr
    gc.collect()

    cat = {"train": [], "test": []}
    cardinalities = []
    for c in cat_cols:
        (c_tr, c_te), card = build_cat_codes(tr[c], [tr[c], test_df[c]])
        cat["train"].append(c_tr); cat["test"].append(c_te)
        cardinalities.append(card)
    cat = {k: np.stack(v, axis=1) for k, v in cat.items()}

    seq = {}
    for name, df in [("train", tr), ("test", test_df)]:
        ts = time.time()
        seq[name] = parse_seq_tail(df["seq"], cfg["SEQ_MAX_LEN"], cfg["SEQ_HASH_BUCKETS"])
        print(f"  seq 파싱 {name}: {time.time()-ts:.0f}s")

    y = {"train": tr[TARGET_COL].to_numpy(dtype=np.float32)}
    print(f"  수치형 {len(num_cols)}개 / 범주형 {len(cat_cols)}개 {dict(zip(cat_cols, cardinalities))}")
    print(f"  준비 완료 ({time.time()-t0:.0f}s)")
    return {"num": num, "cat": cat, "seq": seq, "y": y,
            "num_cols": num_cols, "cat_cols": cat_cols, "cardinalities": cardinalities}


def train_dnn_full(data, cfg, n_epochs):
    """검증 없이 n_epochs만 학습. 스케줄러 T_max는 7-6과 같게 둬서 best epoch까지 같은 LR 궤적을 따름."""
    print("\n" + "=" * 60)
    print(f"[FINAL-DNN] 전체 데이터(day7 포함)로 최종 학습, epochs={n_epochs}")
    print("=" * 60)
    model = WideDeepCTR(len(data["num_cols"]), data["cardinalities"], cfg).to(device)

    y_tr = data["y"]["train"]
    pos_weight_value = (len(y_tr) - y_tr.sum()) / max(y_tr.sum(), 1)
    print(f"  pos_weight={pos_weight_value:.2f}")
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_value], device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["LR"], weight_decay=cfg["WEIGHT_DECAY"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["EPOCHS"])

    rng = np.random.RandomState(cfg["SEED"])
    n_batches = int(np.ceil(len(y_tr) / cfg["BATCH_SIZE"]))
    for epoch in range(1, n_epochs + 1):
        model.train()
        t0, total_loss = time.time(), 0.0
        for b, (num_x, cat_x, seqs, lens, ys) in enumerate(iter_batches(data, "train", cfg["BATCH_SIZE"], True, rng), 1):
            num_x, cat_x, seqs, ys = num_x.to(device), cat_x.to(device), seqs.to(device), ys.to(device)
            optimizer.zero_grad()
            loss = criterion(model(num_x, cat_x, seqs, lens), ys)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(ys)
            if b % 200 == 0:
                print(f"    batch {b}/{n_batches}  loss={loss.item():.4f}  ({time.time()-t0:.0f}s)")
        scheduler.step()
        print(f"[Epoch {epoch}/{n_epochs}] train_loss={total_loss/len(y_tr):.4f}  ({time.time()-t0:.0f}s)")
    return model


if os.path.exists(DNN_FULL_TEST_PRED_PATH):
    dnn_full_test_pred = np.load(DNN_FULL_TEST_PRED_PATH)
    print("DNN 전체 재학습 test 예측은 저장된 .npy에서 불러옴")
else:
    # 7-6 입력 배열은 이미 예측을 .npy로 저장했으니 메모리 확보용으로 해제
    if "dnn_data" in globals():
        del dnn_data
    gc.collect()
    torch.cuda.empty_cache()

    seed_everything(DNN_CFG["SEED"])
    dnn_full_data = prepare_dnn_inputs_full(train, test, feature_cols, DNN_CFG)
    dnn_full_model = train_dnn_full(dnn_full_data, DNN_CFG, DNN_FULL_EPOCHS)
    dnn_full_test_pred = predict_dnn(dnn_full_model, dnn_full_data, "test", DNN_CFG["BATCH_SIZE"] * 4)
    np.save(DNN_FULL_TEST_PRED_PATH, dnn_full_test_pred)
    print(f"  -> {DNN_FULL_TEST_PRED_PATH} 저장 완료")
    del dnn_full_data
    gc.collect()

  DNN train(전체 1~7요일, 100%): 10,704,179  /  test: 1,527,298
  seq 파싱 train: 154s
  seq 파싱 test: 22s
  수치형 123개 / 범주형 6개 {'gender': 3, 'age_group': 9, 'inventory_id': 19, 'day_of_week': 8, 'hour': 25, 'l_feat_14': 3238}
  준비 완료 (207s)

[FINAL-DNN] 전체 데이터(day7 포함)로 최종 학습, epochs=5
  pos_weight=51.43
    batch 200/2614  loss=1.3921  (16s)
    batch 400/2614  loss=1.2418  (30s)
    batch 600/2614  loss=1.3221  (45s)
    batch 800/2614  loss=1.0721  (59s)
    batch 1000/2614  loss=1.2284  (74s)
    batch 1200/2614  loss=1.1704  (89s)
    batch 1400/2614  loss=1.3131  (103s)
    batch 1600/2614  loss=1.0711  (118s)
    batch 1800/2614  loss=1.1401  (133s)
    batch 2000/2614  loss=1.2025  (148s)
    batch 2200/2614  loss=1.2383  (162s)
    batch 2400/2614  loss=1.2252  (177s)
    batch 2600/2614  loss=1.1893  (191s)
[Epoch 1/5] train_loss=1.2007  (192s)
    batch 200/2614  loss=1.1924  (15s)
    batch 400/2614  loss=1.1624  (29s)
    batch 600/2614  loss=1.1165  (44s)
    batch 800/2614  los

In [ ]:
# ------------------------------------------------------------------
# 7-7-3. day7에서 블렌딩 가중치 탐색 -> 같은 가중치로 "전체 재학습" test 예측 블렌딩 후 제출
# ------------------------------------------------------------------
# DNN day7 예측: 메모리에 없으면(런타임 재시작 등) 7-6에서 저장한 .npy 사용
if "dnn_val_pred" not in globals():
    dnn_val_pred = np.load(os.path.join(DATA_DIR, "0923_dnn_day7_val_pred.npy"))
    print("DNN day7 예측은 저장된 .npy에서 불러옴")

assert len(dnn_val_pred) == len(lgb_val_pred) == len(blend_y_valid), "day7 행 수가 다름 -> train을 같은 방식으로 만들었는지 확인"
assert len(dnn_full_test_pred) == len(lgb_full_test_pred) == len(test)


def _logit(p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return np.log(p / (1 - p))


def blend_preds(p_lgb, p_dnn, w_dnn, method):
    """w_dnn = DNN 비중. prob: 확률 가중평균 / logit: logit 가중평균 후 sigmoid."""
    if method == "prob":
        return w_dnn * p_dnn + (1 - w_dnn) * p_lgb
    return 1.0 / (1.0 + np.exp(-(w_dnn * _logit(p_dnn) + (1 - w_dnn) * _logit(p_lgb))))


for name, p in [("LightGBM", lgb_val_pred), ("DNN", dnn_val_pred)]:
    s, ap, wll = competition_score(blend_y_valid, p)
    print(f"[{name:8s} 단독 day7] Score={s:.5f}  AP={ap:.5f}  WLL={wll:.5f}  (평균 예측 {p.mean():.4f})")
print(f"두 모델 day7 예측 상관계수: {np.corrcoef(lgb_val_pred, dnn_val_pred)[0, 1]:.4f}  (낮을수록 블렌딩 효과 기대)")
# 전체 재학습 모델의 test 평균 예측이 day7 검증 모델과 크게 다르면 가중치가 안 맞을 수 있으니 확인용
print(f"test 평균 예측 (전체 재학습): LightGBM {lgb_full_test_pred.mean():.4f} / DNN {dnn_full_test_pred.mean():.4f}")

rows = []
for method in ["prob", "logit"]:
    for w in np.round(np.arange(0, 1.0001, 0.05), 2):
        s, ap, wll = competition_score(blend_y_valid, blend_preds(lgb_val_pred, dnn_val_pred, w, method))
        rows.append({"method": method, "w_dnn": w, "score": s, "ap": ap, "wll": wll})
blend_res = pd.DataFrame(rows)

print("\nday7 Score (행: DNN 비중, 열: 블렌딩 방식)")
print(blend_res.pivot(index="w_dnn", columns="method", values="score").round(5).to_string())

best = blend_res.loc[blend_res["score"].idxmax()]
best_method, best_w = best["method"], float(best["w_dnn"])
lgb_only = blend_res.query("w_dnn == 0 and method == 'prob'")["score"].iloc[0]
print(f"\n>>> 최적: method={best_method}, DNN 비중={best_w:.2f}  "
      f"Score={best['score']:.5f}  AP={best['ap']:.5f}  WLL={best['wll']:.5f}  "
      f"(LightGBM 단독 대비 {best['score'] - lgb_only:+.5f})")
# 주의: 가중치를 day7로 고르고 DNN early stopping도 day7로 했으니 이 점수는 약간 낙관적임.
#       개선폭이 +0.001 미만이면 LB에서 차이 없을 가능성이 큼.

if best_w == 0.0:
    print("DNN을 섞어도 day7 점수가 안 오름 -> 블렌딩 제출 파일은 만들지 않음")
else:
    blend_test_pred = blend_preds(lgb_full_test_pred, dnn_full_test_pred, best_w, best_method)
    BLEND_LGB_DNN_OUTPUT_PATH = os.path.join(
        DATA_DIR, f"0923_blend_lgb_dnn_full_{best_method}_w{int(round(best_w * 100))}_submission.csv")
    sample_sub = pd.read_csv(SAMPLE_SUBMISSION_PATH)
    pred_map = pd.Series(blend_test_pred, index=test[ID_COL].values)
    target_col_in_sample = [c for c in sample_sub.columns if c != ID_COL][0]
    sample_sub[target_col_in_sample] = sample_sub[ID_COL].map(pred_map)
    assert sample_sub[target_col_in_sample].isna().sum() == 0
    sample_sub.to_csv(BLEND_LGB_DNN_OUTPUT_PATH, index=False)
    print(f"  -> {BLEND_LGB_DNN_OUTPUT_PATH} 저장 완료 (shape={sample_sub.shape})")

DNN day7 예측은 저장된 .npy에서 불러옴
[LightGBM 단독 day7] Score=0.34875  AP=0.07277  WLL=0.60071  (평균 예측 0.3872)
[DNN      단독 day7] Score=0.34742  AP=0.07186  WLL=0.60520  (평균 예측 0.3876)
두 모델 day7 예측 상관계수: 0.9229  (낮을수록 블렌딩 효과 기대)
test 평균 예측 (전체 재학습): LightGBM 0.3927 / DNN 0.4045

day7 Score (행: DNN 비중, 열: 블렌딩 방식)
method    logit     prob
w_dnn                   
0.00    0.34875  0.34875
0.05    0.34918  0.34908
0.10    0.34952  0.34936
0.15    0.34977  0.34960
0.20    0.34994  0.34980
0.25    0.35008  0.34996
0.30    0.35017  0.35008
0.35    0.35019  0.35016
0.40    0.35017  0.35020
0.45    0.35011  0.35020
0.50    0.35002  0.35017
0.55    0.34989  0.35009
0.60    0.34972  0.34998
0.65    0.34952  0.34983
0.70    0.34929  0.34963
0.75    0.34904  0.34939
0.80    0.34876  0.34911
0.85    0.34846  0.34879
0.90    0.34813  0.34843
0.95    0.34778  0.34801
1.00    0.34742  0.34742

>>> 최적: method=prob, DNN 비중=0.45  Score=0.35020  AP=0.07490  WLL=0.59871  (LightGBM 단독 대비 +0.00145)
  -> /content/drive

In [ ]:
for v in ["lgb_full_model",
  "lgb_day7_model",
  "dnn_full_model", "dnn_model",
  "models", "final_model"]:
      globals().pop(v, None)
gc.collect();
torch.cuda.empty_cache()

## 8. 완료 알림 (Telegram)

In [ ]:
import requests
from google.colab import userdata

# 실행하면 입력창이 뜨고, 입력하는 글자가 화면에 보이지 않습니다.
BOT_TOKEN = userdata.get('BOT_TOKEN')
CHAT_ID = userdata.get('CHAT_ID')

url = f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage"
payload = {
    'chat_id': CHAT_ID,
    'text': 'colab 다 됐다'
}

response = requests.post(url, data=payload)
print(response.json())

{'ok': True, 'result': {'message_id': 13, 'from': {'id': 8620535929, 'is_bot': True, 'first_name': 'colab_alarmer', 'username': 'colab_alarmerbot'}, 'chat': {'id': 8910305872, 'first_name': 'Maddox', 'last_name': 'Min', 'type': 'private'}, 'date': 1790194100, 'text': 'colab 다 됐다'}}
